# Agent 5 — Symptoms & Mood Agent

Domain agent 5 of 8 in ImmunoSense. Captures the patient's **subjective experience** — symptoms, mood, energy, function. The only agent whose data comes from the patient themselves rather than sensors or labs.

## Special role in the system

Agent 5 plays three roles beyond standard domain-agent duties:

1. **Canonical source of flare events** (hybrid). Agent 5 produces a daily flare_score [0-1] that the Conductor distributes to Agents 1-4 as their `observe_flare()` input. Other agents still run their own trigger detection — Agent 5 just provides the labels.
2. **JEPA modality**. Agent 5 emits a 36-dim dense embedding per day that the World Model consumes for joint-embedding predictive learning.
3. **Closest thing to ground truth**. Patient-reported symptoms are the most direct measure of disease state.

## Three-layer architecture (consistent with Agents 1, 2, 3, 4)

- **Layer 1**: Population baselines — **real NHANES PHQ-8 data** (US general population, ~8,000 adults) + literature-derived disease-specific overrides
- **Layer 2**: Daily summary pipeline that merges voice transcripts (primary), structured forms (higher-confidence backup), and flare button (override)
- **Layer 3**: Per-patient adaptation with robust trackers + BH FDR-corrected trigger detector with biologically pre-registered lags

## Output contract to Conductor

```
SymptomsMoodAgent
  .observe(daily_summary)               # update Layer 3 trackers
  .observe_flare(date, severity)        # rarely called - Agent 5 produces flares
  .analyze() -> SymptomsMoodAgentReport
  .flare_signature(summary) -> dict     # 0-1 wellness deviation
  .daily_flare_score(summary) -> float  # canonical flare label for Conductor
  .jepa_embedding(summary) -> np.array  # 36-dim dense vector
  .raw_hypothesis_evidence() -> list    # ALL 10 hypothesis results, no FDR filter
                                        # for Conductor cross-agent corroboration
```

## Layer 3 design — pre-registered biologically-motivated lags

The trigger detector tests one biologically-motivated lag per feature (10 hypotheses total), rather than scanning all feature × lag combinations (60+ hypotheses). This dramatically improves detection power while preserving FDR control.

Lag categories:

- **Predictive** (lag > 0): feature changes BEFORE flare manifests
  - `sleep_severity` at lag=+2 — bad sleep predicts flare 2 days later
  - `energy_severity` at lag=+1 — low energy predicts flare next day
- **Concurrent** (lag > 0): feature manifests WITH the flare
  - `joint_pain`, `brain_fog_severity`, `gi_distress`, `wellness_severity` at lag=+1
  - `fatigue`, `skin_severity` at lag=+2 (slower-developing symptoms)
- **Reactive** (lag < 0): feature appears AFTER the flare
  - `phq8_score`, `gad7_score` at lag=-3 — mood reacts to recent flare

Lag convention: `feature_on_day(i)` paired with `flare_severity_on_day(i+lag)`. Positive lag → flare follows feature; negative lag → flare preceded feature.

## Known limitations

1. **No real patient daily logs yet**. v1 uses Mock data for Layer 3 validation and a hand-crafted transcript for live Claude API testing. Real integration (mobile app, voice from Agent 8, etc.) is downstream of v1.
2. **Voice transcription errors propagate**. Voice transcripts may contain typos that affect NLP extraction. Structured forms have higher confidence stamps to compensate.
3. **MEM0 stubbed**. Long-term cross-session memory queries are interface-only in v1; real MEM0 swaps in transparently when ready.
4. **Mixed-quality Layer 1 norms**. PHQ-8 baseline IS real — derived from NHANES 2017-March 2020 public data (~8,000 US adults). However, GAD-7 and the 8 symptom severity norms (fatigue, joint pain, etc.) are still approximations from training-data knowledge of published autoimmune cohort literature (LUMINA, CORRONA, NARCOMS). Symptom norms should be calibrated against a real autoimmune cohort dataset (Flaredown, All of Us) in v2.
5. **PHQ-8 not PHQ-9**. Suicide screen (PHQ-9 q9) deferred until proper crisis-flagging infrastructure exists. PHQ-8 is independently validated and excludes that item. NHANES data file contains all 9 items; we sum items 1-8 only.
6. **Subjective severity calibration**. A "7/10 fatigue" means different things to different patients. Layer 3 personalization anchors each patient's personal scale; the flare_score formula uses raw values which assumes some cross-patient calibration. Real-world data will refine the formula.
7. **Agent 5 detector is intentionally conservative**. BH FDR at α=0.10 with only 10 pre-registered hypotheses surfaces only high-confidence patient-level signals. Marginal signals (raw p < 0.05 but BH-suppressed) are intentionally NOT flagged at the agent level — they are exposed via `raw_hypothesis_evidence()` for the Conductor to corroborate against other agents. This separation of concerns means Agent 5 produces few false positives at the agent level; the Conductor handles cross-agent signal aggregation.

In [ ]:
# ============================================================================
# Section 1: Setup
# ============================================================================
import os
import json
import hashlib
import pickle
import random as random_mod
import urllib.request
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional, Protocol

import numpy as np
import pandas as pd
from scipy.stats import norm

from dotenv import load_dotenv

ENV_PATH = Path.cwd().parent / '.env' if Path.cwd().name == 'notebooks' else Path.cwd() / '.env'
load_dotenv(ENV_PATH)

def _key_status(name: str) -> str:
    val = os.environ.get(name)
    return f'OK ({len(val)} chars)' if val else 'NOT SET'

print('API key status:')
print(f'  ANTHROPIC_API_KEY: {_key_status("ANTHROPIC_API_KEY")}')

ARTIFACT_DIR = Path('./artifacts/agent5')
CACHE_DIR = ARTIFACT_DIR / 'nlp_cache'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'\nArtifact dir: {ARTIFACT_DIR.resolve()}')
print(f'.env exists: {ENV_PATH.exists()}')

In [ ]:
# ============================================================================
# Section 2: Layer 1 - Population Baseline Tables
#
# Disease-stratified literature-derived (mean, std) for each feature.
# Note: 'Mixed' PHQ-8 will be OVERRIDDEN with real NHANES values in Section 3.
# Disease-specific PHQ-8 and ALL other features remain literature approximations.
# ============================================================================

SYMPTOM_FEATURES = [
    'fatigue', 'joint_pain', 'brain_fog_severity', 'gi_distress',
    'skin_severity', 'sleep_severity', 'energy_severity', 'wellness_severity',
]
MOOD_FEATURES = ['phq8_score', 'gad7_score']
ALL_FEATURES = SYMPTOM_FEATURES + MOOD_FEATURES

DISEASE_TYPES = ['RA', 'SLE', 'MS', 'Sjogrens', 'PsA', 'Mixed']

DISEASE_NORMS = {
    'RA': {
        'fatigue':             (5.8, 2.0),  'joint_pain':         (5.5, 2.3),
        'brain_fog_severity':  (3.2, 2.0),  'gi_distress':        (2.5, 1.8),
        'skin_severity':       (2.0, 1.5),  'sleep_severity':     (4.5, 2.2),
        'energy_severity':     (5.5, 2.1),  'wellness_severity':  (5.0, 2.0),
        'phq8_score':          (7.5, 4.5),  'gad7_score':         (6.0, 4.0),
    },
    'SLE': {
        'fatigue':             (6.4, 2.1),  'joint_pain':         (4.8, 2.5),
        'brain_fog_severity':  (5.1, 2.2),  'gi_distress':        (3.0, 2.0),
        'skin_severity':       (4.5, 2.5),  'sleep_severity':     (5.0, 2.3),
        'energy_severity':     (6.0, 2.2),  'wellness_severity':  (5.5, 2.1),
        'phq8_score':          (8.0, 4.8),  'gad7_score':         (6.5, 4.2),
    },
    'MS': {
        'fatigue':             (6.1, 2.2),  'joint_pain':         (3.5, 2.0),
        'brain_fog_severity':  (5.5, 2.0),  'gi_distress':        (3.5, 2.2),
        'skin_severity':       (2.0, 1.5),  'sleep_severity':     (5.2, 2.3),
        'energy_severity':     (6.2, 2.0),  'wellness_severity':  (5.3, 2.2),
        'phq8_score':          (7.8, 4.6),  'gad7_score':         (6.2, 4.1),
    },
    'Sjogrens': {
        'fatigue':             (6.8, 2.0),  'joint_pain':         (4.5, 2.4),
        'brain_fog_severity':  (4.8, 2.1),  'gi_distress':        (4.0, 2.3),
        'skin_severity':       (3.0, 2.0),  'sleep_severity':     (5.5, 2.2),
        'energy_severity':     (6.5, 2.0),  'wellness_severity':  (5.8, 2.0),
        'phq8_score':          (8.2, 4.7),  'gad7_score':         (6.8, 4.3),
    },
    'PsA': {
        'fatigue':             (5.5, 2.1),  'joint_pain':         (5.8, 2.4),
        'brain_fog_severity':  (3.0, 1.8),  'gi_distress':        (2.5, 1.8),
        'skin_severity':       (5.5, 2.5),  'sleep_severity':     (4.8, 2.2),
        'energy_severity':     (5.2, 2.1),  'wellness_severity':  (4.8, 2.0),
        'phq8_score':          (7.0, 4.3),  'gad7_score':         (5.8, 3.9),
    },
    'Mixed': {  # General/unknown - 'Mixed' PHQ-8 will be replaced with NHANES below
        'fatigue':             (5.8, 2.3),  'joint_pain':         (4.5, 2.5),
        'brain_fog_severity':  (4.0, 2.3),  'gi_distress':        (3.2, 2.2),
        'skin_severity':       (3.5, 2.5),  'sleep_severity':     (5.0, 2.4),
        'energy_severity':     (5.8, 2.2),  'wellness_severity':  (5.2, 2.2),
        'phq8_score':          (7.5, 4.7),  'gad7_score':         (6.2, 4.2),
    },
}

missing = [(d, f) for d in DISEASE_TYPES for f in ALL_FEATURES
           if f not in DISEASE_NORMS[d]]
assert not missing, f'Missing norm cells: {missing}'

print(f'Disease-stratified norms: {len(DISEASE_TYPES)} diseases x {len(ALL_FEATURES)} features = {len(DISEASE_TYPES)*len(ALL_FEATURES)} cells')
print('Note: Mixed/PHQ-8 will be overridden with NHANES real data in Section 3.')

In [ ]:
# ============================================================================
# Section 3: NHANES Integration - Real PHQ-8 Baseline
#
# Downloads NHANES P_DPQ.XPT (Public Use, 2017-March 2020 pre-pandemic combined
# cycle, adults 18+), computes PHQ-8 distribution from real US population data,
# and OVERRIDES the literature-derived PHQ-8 value in DISEASE_NORMS['Mixed']
# with the real population statistics.
#
# CRITICAL: filter on items 1-8 ONLY (drop item 9 suicide screen). NHANES
# skip-logic means many respondents have missing values on item 9 that aren't
# real refusals - filtering on all 9 items would eliminate most valid responses.
# ============================================================================

NHANES_DPQ_URL = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DPQ.XPT'
NHANES_DATA_DIR = Path('./data/nhanes') if Path('./data').exists() else Path('../data/nhanes')
NHANES_DATA_DIR.mkdir(parents=True, exist_ok=True)
NHANES_DPQ_LOCAL = NHANES_DATA_DIR / 'P_DPQ.XPT'


def download_nhanes_dpq(force: bool = False) -> Path:
    """Download NHANES depression screening file, or return cached copy."""
    if NHANES_DPQ_LOCAL.exists() and not force:
        size_kb = NHANES_DPQ_LOCAL.stat().st_size / 1024
        print(f'NHANES P_DPQ.XPT already cached ({size_kb:.0f} KB)')
        return NHANES_DPQ_LOCAL

    print(f'Downloading NHANES P_DPQ from CDC ...')
    try:
        urllib.request.urlretrieve(NHANES_DPQ_URL, NHANES_DPQ_LOCAL)
        size_kb = NHANES_DPQ_LOCAL.stat().st_size / 1024
        print(f'  Done ({size_kb:.0f} KB)')
        return NHANES_DPQ_LOCAL
    except Exception as e:
        if NHANES_DPQ_LOCAL.exists():
            print(f'  Download failed ({e}); using existing cache')
            return NHANES_DPQ_LOCAL
        raise RuntimeError(f'NHANES download failed and no cache: {e}')


def compute_phq8_norms_from_nhanes(local_path: Path) -> dict:
    """Parse NHANES DPQ file, compute PHQ-8 distribution statistics.

    Filters on items 1-8 ONLY (DPQ010 through DPQ080), deliberately dropping
    item 9 (suicide screen). NHANES skip-logic causes many respondents to have
    missing values on item 9 that aren't real refusals - including that filter
    would eliminate most valid PHQ-8 responses.
    """
    df = pd.read_sas(local_path, format='xport')
    phq_items_all = [f'DPQ{i:03d}' for i in range(10, 100, 10)]
    phq8_items = phq_items_all[:8]   # items 1-8 only

    missing = [c for c in phq8_items if c not in df.columns]
    if missing:
        raise ValueError(f'NHANES DPQ missing columns: {missing}')

    # Filter to valid responses (0-3) on items 1-8 only
    valid = df.copy()
    for c in phq8_items:
        valid = valid[valid[c].isin([0, 1, 2, 3])]
    if len(valid) == 0:
        raise ValueError('No valid PHQ-8 responses found')

    valid = valid.copy()
    valid['phq8_total'] = valid[phq8_items].sum(axis=1)

    def cat(s):
        if s <= 4: return 'minimal'
        if s <= 9: return 'mild'
        if s <= 14: return 'moderate'
        if s <= 19: return 'mod_severe'
        return 'severe'

    return {
        'mean': float(valid['phq8_total'].mean()),
        'std': float(valid['phq8_total'].std()),
        'median': float(valid['phq8_total'].median()),
        'p25': float(valid['phq8_total'].quantile(0.25)),
        'p75': float(valid['phq8_total'].quantile(0.75)),
        'n': len(valid),
        'severity_pct': valid['phq8_total'].apply(cat).value_counts(normalize=True).to_dict(),
        'source': 'NHANES 2017-March 2020 pre-pandemic, adults 18+',
    }


def update_disease_norms_with_nhanes(verbose: bool = True) -> dict:
    """Download NHANES, compute PHQ-8 norms, update DISEASE_NORMS['Mixed']."""
    try:
        local = download_nhanes_dpq()
        stats = compute_phq8_norms_from_nhanes(local)
    except Exception as e:
        print(f'WARNING: Could not load NHANES PHQ-8 data: {e}')
        print(f'Falling back to literature-approximated value for Mixed/PHQ-8.')
        return {'error': str(e)}

    if verbose:
        print(f'NHANES PHQ-8 statistics (N={stats["n"]:,}):')
        print(f'  Mean:   {stats["mean"]:.2f}')
        print(f'  Std:    {stats["std"]:.2f}')
        print(f'  Median: {stats["median"]:.1f}')
        print(f'  IQR:    [{stats["p25"]:.1f}, {stats["p75"]:.1f}]')
        print(f'  Severity distribution:')
        for sev in ['minimal', 'mild', 'moderate', 'mod_severe', 'severe']:
            pct = stats['severity_pct'].get(sev, 0) * 100
            print(f'    {sev:12s}: {pct:.1f}%')

    old_value = DISEASE_NORMS['Mixed']['phq8_score']
    new_value = (round(stats['mean'], 2), round(stats['std'], 2))
    DISEASE_NORMS['Mixed']['phq8_score'] = new_value

    if verbose:
        print(f'\nDISEASE_NORMS["Mixed"]["phq8_score"]:')
        print(f'  Before (literature):  {old_value}')
        print(f'  After  (NHANES real): {new_value}')

    return stats


nhanes_phq8_stats = update_disease_norms_with_nhanes(verbose=True)

In [ ]:
# ============================================================================
# Section 4: Layer 1 - Percentile Lookup API
# ============================================================================

DISEASE_ALIASES = {
    'rheumatoid arthritis': 'RA',     'rheumatoid': 'RA',         'ra': 'RA',
    'lupus': 'SLE',                    'systemic lupus': 'SLE',    'sle': 'SLE',
    'multiple sclerosis': 'MS',        'ms': 'MS',
    "sjogren's": 'Sjogrens',           'sjogrens': 'Sjogrens',     'sjogren': 'Sjogrens',
    'psoriatic arthritis': 'PsA',      'psa': 'PsA',
    'unknown': 'Mixed',                'multiple': 'Mixed',         '': 'Mixed',
}

def normalize_disease(disease) -> str:
    if disease is None:
        return 'Mixed'
    key = str(disease).strip().lower()
    if key in DISEASE_ALIASES:
        return DISEASE_ALIASES[key]
    for d in DISEASE_TYPES:
        if d.lower() == key:
            return d
    return 'Mixed'


def get_population_percentile(disease: str, feature: str, value: float) -> float:
    """Place observed value on disease-stratified population CDF."""
    norm_d = normalize_disease(disease)
    if feature not in DISEASE_NORMS[norm_d]:
        raise ValueError(f"Unknown feature '{feature}'")
    mean, std = DISEASE_NORMS[norm_d][feature]
    if std <= 0:
        return 0.5
    return float(norm.cdf((value - mean) / std))


# Smoke tests
assert normalize_disease('Rheumatoid Arthritis') == 'RA'
assert normalize_disease('lupus') == 'SLE'
assert normalize_disease(None) == 'Mixed'
assert abs(get_population_percentile('RA', 'fatigue', 5.8) - 0.5) < 0.01
print('Layer 1 functions defined and smoke-tested.')

In [ ]:
# ============================================================================
# Section 5: Data Source Contracts
# ============================================================================
@dataclass
class FetchedSymptoms:
    fatigue: Optional[float] = None
    joint_pain: Optional[float] = None
    brain_fog_severity: Optional[float] = None
    gi_distress: Optional[float] = None
    skin_severity: Optional[float] = None
    sleep_severity: Optional[float] = None
    energy_severity: Optional[float] = None
    wellness_severity: Optional[float] = None
    phq8_score: Optional[float] = None
    gad7_score: Optional[float] = None
    emotional_valence: Optional[float] = None
    new_symptom_mentions: list = field(default_factory=list)
    explicit_flare: Optional[bool] = None
    explicit_flare_severity: Optional[float] = None
    confidence: dict = field(default_factory=dict)
    sources: dict = field(default_factory=dict)
    errors: list = field(default_factory=list)


class SymptomDataSource(Protocol):
    def fetch(self, patient_id: str, target_date: str) -> FetchedSymptoms: ...


@dataclass
class DailySymptomMoodSummary:
    date: str
    patient_id: str
    disease: str
    fatigue: Optional[float]
    joint_pain: Optional[float]
    brain_fog_severity: Optional[float]
    gi_distress: Optional[float]
    skin_severity: Optional[float]
    sleep_severity: Optional[float]
    energy_severity: Optional[float]
    wellness_severity: Optional[float]
    phq8_score: Optional[float]
    gad7_score: Optional[float]
    emotional_valence: Optional[float] = None
    new_symptom_mentions: list = field(default_factory=list)
    percentiles: dict = field(default_factory=dict)
    threshold_alerts: dict = field(default_factory=dict)
    feature_confidence: dict = field(default_factory=dict)
    sources: dict = field(default_factory=dict)
    errors: list = field(default_factory=list)
    overall_confidence: float = 0.0
    flare_score: float = 0.0
    flare_button_pressed: bool = False

print('Data source contracts defined.')

In [ ]:
# ============================================================================
# Section 6: Clinical Threshold Classifiers
# ============================================================================

def classify_phq8(score: Optional[float]) -> Optional[str]:
    if score is None: return None
    if score <= 4:  return 'minimal'
    if score <= 9:  return 'mild'
    if score <= 14: return 'moderate'
    if score <= 19: return 'moderately_severe'
    return 'severe'


def classify_gad7(score: Optional[float]) -> Optional[str]:
    if score is None: return None
    if score <= 4:  return 'minimal'
    if score <= 9:  return 'mild'
    if score <= 14: return 'moderate'
    return 'severe'


def classify_symptom_severity(value: Optional[float]) -> Optional[str]:
    if value is None: return None
    if value <= 3:  return 'mild'
    if value <= 6:  return 'moderate'
    return 'severe'


def classify_all_thresholds(fetched: FetchedSymptoms) -> dict:
    return {
        'fatigue':             classify_symptom_severity(fetched.fatigue),
        'joint_pain':          classify_symptom_severity(fetched.joint_pain),
        'brain_fog':           classify_symptom_severity(fetched.brain_fog_severity),
        'gi_distress':         classify_symptom_severity(fetched.gi_distress),
        'skin':                classify_symptom_severity(fetched.skin_severity),
        'sleep':               classify_symptom_severity(fetched.sleep_severity),
        'energy':              classify_symptom_severity(fetched.energy_severity),
        'wellness':            classify_symptom_severity(fetched.wellness_severity),
        'phq8':                classify_phq8(fetched.phq8_score),
        'gad7':                classify_gad7(fetched.gad7_score),
    }

print('Clinical threshold classifiers defined.')

In [ ]:
# ============================================================================
# Section 7: Daily Flare Score - the canonical Agent 5 -> Conductor signal
# ============================================================================
FLARE_WEIGHTS = {
    'fatigue':             0.20,
    'joint_pain':          0.20,
    'brain_fog_severity':  0.15,
    'gi_distress':         0.10,
    'skin_severity':       0.10,
    'sleep_severity':      0.10,
    'energy_severity':     0.10,
    'wellness_severity':   0.05,
}
assert abs(sum(FLARE_WEIGHTS.values()) - 1.0) < 1e-9


def compute_daily_flare_score(
    fetched: FetchedSymptoms,
    button_override_floor: float = 0.8,
) -> float:
    score = 0.0
    for feat, w in FLARE_WEIGHTS.items():
        v = getattr(fetched, feat, None)
        if v is not None:
            score += w * (max(0.0, min(10.0, v)) / 10.0)

    if fetched.explicit_flare:
        explicit_sev = fetched.explicit_flare_severity if fetched.explicit_flare_severity is not None else 0.85
        score = max(score, button_override_floor, explicit_sev)

    return min(1.0, max(0.0, score))

print('compute_daily_flare_score defined.')

In [ ]:
# ============================================================================
# Section 8: MockSymptomSource (synthetic data for Layer 3 validation)
# ============================================================================
class MockSymptomSource:
    def __init__(self, disease: str = 'Mixed', seed_offset: int = 0):
        self.disease = normalize_disease(disease)
        self.seed_offset = seed_offset

    def fetch(self, patient_id: str, target_date: str) -> FetchedSymptoms:
        seed_key = f'{patient_id},{target_date},{self.seed_offset}'
        seed = int(hashlib.md5(seed_key.encode()).hexdigest()[:8], 16)
        rng = random_mod.Random(seed)

        norms = DISEASE_NORMS[self.disease]

        def _sample(feat, lo=0.0, hi=10.0):
            mean, std = norms[feat]
            return max(lo, min(hi, rng.gauss(mean, std)))

        result = FetchedSymptoms(
            fatigue=round(_sample('fatigue'), 1),
            joint_pain=round(_sample('joint_pain'), 1),
            brain_fog_severity=round(_sample('brain_fog_severity'), 1),
            gi_distress=round(_sample('gi_distress'), 1),
            skin_severity=round(_sample('skin_severity'), 1),
            sleep_severity=round(_sample('sleep_severity'), 1),
            energy_severity=round(_sample('energy_severity'), 1),
            wellness_severity=round(_sample('wellness_severity'), 1),
            phq8_score=round(max(0, min(24, rng.gauss(*norms['phq8_score']))), 0),
            gad7_score=round(max(0, min(21, rng.gauss(*norms['gad7_score']))), 0),
            explicit_flare=False,
        )
        for f in ALL_FEATURES:
            result.confidence[f] = 'synthetic'
            result.sources[f] = 'mock'
        return result

print('MockSymptomSource defined.')

In [ ]:
# ============================================================================
# Section 9: VoiceTranscriptSource (real Claude API extraction)
# ============================================================================
try:
    import anthropic
    _ANTHROPIC_AVAILABLE = True
except ImportError:
    _ANTHROPIC_AVAILABLE = False
    print('Note: anthropic SDK not installed. VoiceTranscriptSource will error if used.')


_EXTRACTION_PROMPT = '''You are a clinical NLP assistant extracting structured symptom data from a patient's voice transcript.

The patient has an autoimmune condition and is describing how they feel today.

Extract values for these features. Use null (not 0) when the feature is not mentioned. Be conservative - do not infer high severities from vague language.

Severity scale 0-10 (higher = worse):
- fatigue: how tired/exhausted (0=none, 10=cannot function)
- joint_pain: aggregate joint pain
- brain_fog_severity: cognitive impairment (0=clear-headed, 10=cannot think)
- gi_distress: nausea, abdominal pain, bowel issues
- skin_severity: rash, itching, lesions
- sleep_severity: poor sleep quality last night (0=excellent sleep, 10=no sleep)
- energy_severity: low energy (0=normal energy, 10=no energy)
- wellness_severity: overall feeling unwell (0=feel great, 10=feel terrible)

Mood (use null if not assessable from text):
- phq8_score: 0-24 estimate of depression severity (only if patient describes depression-related symptoms)
- gad7_score: 0-21 estimate of anxiety severity (only if patient describes anxiety)

Other:
- emotional_valence: -1 (very negative tone) to +1 (very positive tone), null if neutral or unclear
- new_symptom_mentions: list of any specific symptoms mentioned that aren't in the 8 above (e.g., "Raynaud's", "mouth ulcers", "hair loss")
- explicit_flare: true ONLY if patient explicitly says they are having a flare

Respond with ONLY valid JSON matching this schema (no prose, no markdown):
{
  "fatigue": null or number,
  "joint_pain": null or number,
  "brain_fog_severity": null or number,
  "gi_distress": null or number,
  "skin_severity": null or number,
  "sleep_severity": null or number,
  "energy_severity": null or number,
  "wellness_severity": null or number,
  "phq8_score": null or number,
  "gad7_score": null or number,
  "emotional_valence": null or number,
  "new_symptom_mentions": [list of strings],
  "explicit_flare": true or false
}

Patient transcript:
"""
{TRANSCRIPT}
"""'''


class VoiceTranscriptSource:
    """Extracts structured symptoms from voice transcript via Claude Haiku."""

    def __init__(self,
                 api_key: Optional[str] = None,
                 cache_dir: Optional[Path] = None,
                 model: str = 'claude-haiku-4-5-20251001'):
        self.api_key = api_key or os.environ.get('ANTHROPIC_API_KEY')
        self.cache_dir = cache_dir or CACHE_DIR
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.model = model
        self._client = None

    def _get_client(self):
        if not _ANTHROPIC_AVAILABLE:
            raise RuntimeError('anthropic SDK not installed')
        if not self.api_key:
            raise RuntimeError('ANTHROPIC_API_KEY not set')
        if self._client is None:
            self._client = anthropic.Anthropic(api_key=self.api_key)
        return self._client

    def extract(self, transcript: str, patient_id: str, target_date: str) -> FetchedSymptoms:
        if not transcript or not transcript.strip():
            return FetchedSymptoms(errors=['VoiceTranscript: empty input'])

        text_hash = hashlib.md5(transcript.encode()).hexdigest()[:12]
        cache_path = self.cache_dir / f'voice_{patient_id}_{target_date}_{text_hash}.pkl'
        if cache_path.exists():
            with open(cache_path, 'rb') as f:
                return pickle.load(f)

        try:
            client = self._get_client()
        except RuntimeError as e:
            return FetchedSymptoms(errors=[f'VoiceTranscript setup: {e}'])

        prompt = _EXTRACTION_PROMPT.replace('{TRANSCRIPT}', transcript)

        try:
            response = client.messages.create(
                model=self.model,
                max_tokens=1024,
                messages=[{'role': 'user', 'content': prompt}],
            )
            raw_text = response.content[0].text.strip()
        except Exception as e:
            return FetchedSymptoms(errors=[f'VoiceTranscript API: {type(e).__name__}: {e}'])

        if raw_text.startswith('```'):
            lines = raw_text.split('\n')
            raw_text = '\n'.join(lines[1:-1]) if lines[-1].startswith('```') else '\n'.join(lines[1:])
        try:
            data = json.loads(raw_text)
        except json.JSONDecodeError as e:
            return FetchedSymptoms(errors=[f'VoiceTranscript JSON parse: {e}', f'raw: {raw_text[:200]}'])

        result = FetchedSymptoms(
            fatigue=data.get('fatigue'),
            joint_pain=data.get('joint_pain'),
            brain_fog_severity=data.get('brain_fog_severity'),
            gi_distress=data.get('gi_distress'),
            skin_severity=data.get('skin_severity'),
            sleep_severity=data.get('sleep_severity'),
            energy_severity=data.get('energy_severity'),
            wellness_severity=data.get('wellness_severity'),
            phq8_score=data.get('phq8_score'),
            gad7_score=data.get('gad7_score'),
            emotional_valence=data.get('emotional_valence'),
            new_symptom_mentions=data.get('new_symptom_mentions', []),
            explicit_flare=data.get('explicit_flare', False),
        )

        for feat in ALL_FEATURES:
            if getattr(result, feat) is not None:
                result.confidence[feat] = 'voice_extracted'
                result.sources[feat] = 'claude-haiku'

        try:
            with open(cache_path, 'wb') as f:
                pickle.dump(result, f)
        except Exception as e:
            result.errors.append(f'VoiceTranscript cache write: {e}')

        return result

print('VoiceTranscriptSource defined.')

In [ ]:
# ============================================================================
# Section 10: StructuredFormSource (direct slider/checkbox input)
# ============================================================================
class StructuredFormSource:
    def from_dict(self, form_data: dict) -> FetchedSymptoms:
        result = FetchedSymptoms(
            fatigue=form_data.get('fatigue'),
            joint_pain=form_data.get('joint_pain'),
            brain_fog_severity=form_data.get('brain_fog_severity'),
            gi_distress=form_data.get('gi_distress'),
            skin_severity=form_data.get('skin_severity'),
            sleep_severity=form_data.get('sleep_severity'),
            energy_severity=form_data.get('energy_severity'),
            wellness_severity=form_data.get('wellness_severity'),
            phq8_score=form_data.get('phq8_score'),
            gad7_score=form_data.get('gad7_score'),
            explicit_flare=form_data.get('explicit_flare', False),
            explicit_flare_severity=form_data.get('explicit_flare_severity'),
        )
        for feat in ALL_FEATURES:
            if getattr(result, feat) is not None:
                result.confidence[feat] = 'structured'
                result.sources[feat] = 'form'
        return result

print('StructuredFormSource defined.')

In [ ]:
# ============================================================================
# Section 11: FlareButtonSource (explicit override signal)
# ============================================================================
class FlareButtonSource:
    def from_event(self, severity: Optional[float] = None) -> FetchedSymptoms:
        result = FetchedSymptoms(
            explicit_flare=True,
            explicit_flare_severity=severity,
        )
        result.confidence['explicit_flare'] = 'flare_button'
        result.sources['explicit_flare'] = 'patient_button'
        return result

print('FlareButtonSource defined.')

In [ ]:
# ============================================================================
# Section 12: CompositeSymptomSource (production source)
# ============================================================================
class CompositeSymptomSource:
    def __init__(self,
                 voice: Optional['VoiceTranscriptSource'] = None,
                 structured: Optional['StructuredFormSource'] = None,
                 flare_button: Optional['FlareButtonSource'] = None,
                 mock_fallback: Optional['MockSymptomSource'] = None):
        self.voice = voice
        self.structured = structured or StructuredFormSource()
        self.flare_button = flare_button or FlareButtonSource()
        self.mock_fallback = mock_fallback

    def assemble(self,
                 patient_id: str,
                 target_date: str,
                 transcript: Optional[str] = None,
                 form_data: Optional[dict] = None,
                 flare_event_severity: Optional[float] = None,
                 ) -> FetchedSymptoms:
        merged = FetchedSymptoms()

        # Voice extraction
        if transcript and self.voice is not None:
            voice_result = self.voice.extract(transcript, patient_id, target_date)
            for feat in ALL_FEATURES:
                v = getattr(voice_result, feat)
                if v is not None:
                    setattr(merged, feat, v)
                    merged.confidence[feat] = 'voice_extracted'
                    merged.sources[feat] = 'claude-haiku'
            merged.emotional_valence = voice_result.emotional_valence
            merged.new_symptom_mentions = list(voice_result.new_symptom_mentions)
            merged.errors.extend(voice_result.errors)
            if voice_result.explicit_flare:
                merged.explicit_flare = True

        # Structured form (overrides voice)
        if form_data:
            structured_result = self.structured.from_dict(form_data)
            for feat in ALL_FEATURES:
                v = getattr(structured_result, feat)
                if v is not None:
                    setattr(merged, feat, v)
                    merged.confidence[feat] = 'structured'
                    merged.sources[feat] = 'form'
            if structured_result.explicit_flare:
                merged.explicit_flare = True
                if structured_result.explicit_flare_severity is not None:
                    merged.explicit_flare_severity = structured_result.explicit_flare_severity

        # Flare button
        if flare_event_severity is not None:
            button_result = self.flare_button.from_event(severity=flare_event_severity)
            merged.explicit_flare = True
            merged.explicit_flare_severity = max(
                merged.explicit_flare_severity or 0.0,
                button_result.explicit_flare_severity or 0.0
            )
            merged.confidence['explicit_flare'] = 'flare_button'
            merged.sources['explicit_flare'] = 'patient_button'

        # Mock fallback
        if self.mock_fallback is not None:
            mock_result = self.mock_fallback.fetch(patient_id, target_date)
            for feat in ALL_FEATURES:
                if getattr(merged, feat) is None:
                    setattr(merged, feat, getattr(mock_result, feat))
                    merged.confidence[feat] = 'synthetic'
                    merged.sources[feat] = 'mock'

        return merged

print('CompositeSymptomSource defined.')

In [ ]:
# ============================================================================
# Section 13: Layer 2 Pipeline - process_symptom_day
# ============================================================================
_NON_SYNTHETIC = {'structured', 'voice_extracted', 'flare_button'}


def process_symptom_day(
    patient_id: str,
    target_date: str,
    disease: str,
    transcript: Optional[str] = None,
    form_data: Optional[dict] = None,
    flare_event_severity: Optional[float] = None,
    composite_source: Optional[CompositeSymptomSource] = None,
) -> DailySymptomMoodSummary:
    if composite_source is None:
        composite_source = CompositeSymptomSource(voice=VoiceTranscriptSource())

    fetched = composite_source.assemble(
        patient_id=patient_id,
        target_date=target_date,
        transcript=transcript,
        form_data=form_data,
        flare_event_severity=flare_event_severity,
    )

    norm_disease = normalize_disease(disease)

    percentiles = {}
    for feat in ALL_FEATURES:
        v = getattr(fetched, feat)
        percentiles[feat] = (None if v is None
                             else get_population_percentile(norm_disease, feat, v))

    threshold_alerts = classify_all_thresholds(fetched)
    flare_score = compute_daily_flare_score(fetched)

    real_count = sum(1 for feat in ALL_FEATURES
                     if fetched.confidence.get(feat) in _NON_SYNTHETIC)
    overall_confidence = real_count / 10.0

    return DailySymptomMoodSummary(
        date=target_date,
        patient_id=patient_id,
        disease=norm_disease,
        fatigue=fetched.fatigue,
        joint_pain=fetched.joint_pain,
        brain_fog_severity=fetched.brain_fog_severity,
        gi_distress=fetched.gi_distress,
        skin_severity=fetched.skin_severity,
        sleep_severity=fetched.sleep_severity,
        energy_severity=fetched.energy_severity,
        wellness_severity=fetched.wellness_severity,
        phq8_score=fetched.phq8_score,
        gad7_score=fetched.gad7_score,
        emotional_valence=fetched.emotional_valence,
        new_symptom_mentions=list(fetched.new_symptom_mentions),
        percentiles=percentiles,
        threshold_alerts=threshold_alerts,
        feature_confidence=dict(fetched.confidence),
        sources=dict(fetched.sources),
        errors=list(fetched.errors),
        overall_confidence=overall_confidence,
        flare_score=flare_score,
        flare_button_pressed=bool(fetched.explicit_flare),
    )

print('Layer 2 pipeline defined.')

In [ ]:
# ============================================================================
# Section 14: JEPA Embedding (36-dim daily dense vector)
# ============================================================================
def compute_jepa_embedding(summary: DailySymptomMoodSummary) -> np.ndarray:
    vec = np.zeros(36, dtype=np.float32)

    for i, feat in enumerate(SYMPTOM_FEATURES):
        v = getattr(summary, feat, None)
        vec[i] = (max(0.0, min(10.0, v)) / 10.0) if v is not None else 0.0

    vec[8] = (summary.phq8_score / 24.0) if summary.phq8_score is not None else 0.0
    vec[9] = (summary.gad7_score / 21.0) if summary.gad7_score is not None else 0.0

    if summary.disease in DISEASE_TYPES:
        vec[10 + DISEASE_TYPES.index(summary.disease)] = 1.0

    for i, feat in enumerate(ALL_FEATURES):
        p = summary.percentiles.get(feat)
        vec[16 + i] = float(p) if p is not None else 0.0

    vec[26] = (summary.emotional_valence + 1.0) / 2.0 if summary.emotional_valence is not None else 0.5
    vec[27] = min(1.0, len(summary.new_symptom_mentions) / 5.0)
    vec[28] = float(summary.flare_score)
    vec[29] = float(summary.overall_confidence)

    sources_seen = set(summary.feature_confidence.values())
    vec[30] = 1.0 if 'structured' in sources_seen else 0.0
    vec[31] = 1.0 if 'voice_extracted' in sources_seen else 0.0
    vec[32] = 1.0 if 'flare_button' in sources_seen else 0.0
    vec[33] = 1.0 if 'synthetic' in sources_seen else 0.0

    return vec

print('compute_jepa_embedding defined (36-dim).')

In [ ]:
# ============================================================================
# Section 15: MEM0 Stub Interface
# ============================================================================
@dataclass
class HistoricalPattern:
    pattern_id: str
    similarity: float
    summary: str
    date_range: tuple


class MemoryStore(Protocol):
    def query_patterns(self, patient_id: str, symptom_cluster: dict, top_k: int = 3) -> list: ...
    def add_observation(self, patient_id: str, summary: 'DailySymptomMoodSummary') -> None: ...


class StubMemoryStore:
    """In-memory placeholder until MEM0 exists."""

    def __init__(self):
        self._log = {}

    def query_patterns(self, patient_id: str, symptom_cluster: dict, top_k: int = 3) -> list:
        return []

    def add_observation(self, patient_id: str, summary: 'DailySymptomMoodSummary') -> None:
        self._log.setdefault(patient_id, []).append(summary)

    def n_observations(self, patient_id: str) -> int:
        return len(self._log.get(patient_id, []))

print('MEM0 stub interface defined.')

In [ ]:
# ============================================================================
# Section 16: Layer 3 - Robust Baseline Trackers
# ============================================================================
class _UnivariateRobustTracker:
    def __init__(self, window=14, anomaly_threshold=2.0, personalization_days=25):
        self.window = window
        self.anomaly_threshold = anomaly_threshold
        self.personalization_days = personalization_days
        self.clean_history = deque(maxlen=window)

    def update(self, value):
        if value is None or pd.isna(value):
            return
        if len(self.clean_history) < 3:
            self.clean_history.append(float(value))
        else:
            arr = np.array(self.clean_history)
            med = np.median(arr)
            q75, q25 = np.percentile(arr, [75, 25])
            iqr = q75 - q25
            if iqr < 1e-9:
                self.clean_history.append(float(value))
            elif abs(value - med) / iqr < self.anomaly_threshold:
                self.clean_history.append(float(value))

    def baseline(self):
        n = len(self.clean_history)
        if n < 3:
            return {'median': float('nan'), 'iqr': float('nan'),
                    'n_clean': n, 'personal_weight': 0.0}
        arr = np.array(self.clean_history)
        q75, q25 = np.percentile(arr, [75, 25])
        return {
            'median': float(np.median(arr)),
            'iqr': float(q75 - q25),
            'n_clean': n,
            'personal_weight': min(1.0, n / self.personalization_days),
        }

    def anomaly_score(self, value):
        if pd.isna(value) or len(self.clean_history) < 3:
            return float('nan')
        arr = np.array(self.clean_history)
        med = np.median(arr)
        q75, q25 = np.percentile(arr, [75, 25])
        iqr = q75 - q25
        if iqr < 1e-9:
            return 0.0
        return float((value - med) / iqr)


class SymptomsMoodRobustTracker:
    def __init__(self, window=14, anomaly_threshold=2.0, personalization_days=25):
        self.trackers = {
            f: _UnivariateRobustTracker(window, anomaly_threshold, personalization_days)
            for f in ALL_FEATURES
        }

    def update(self, daily_summary):
        for f in ALL_FEATURES:
            v = getattr(daily_summary, f, None)
            if v is not None:
                self.trackers[f].update(v)

    def report(self):
        return {f: t.baseline() for f, t in self.trackers.items()}

    def anomaly_scores(self, daily_summary):
        return {f: self.trackers[f].anomaly_score(getattr(daily_summary, f, float('nan')))
                for f in ALL_FEATURES}

print('SymptomsMoodRobustTracker defined (10 features).')

In [ ]:
# ============================================================================
# Section 17: Layer 3 - Trigger Detector
#
# Pre-registered lag per feature based on biology. 10 hypotheses total.
# BH FDR at alpha=0.10 for high-confidence pattern detection.
# Additionally exposes raw_hypothesis_evidence() for Conductor cross-agent corroboration.
#
# Lag convention: feature_on_day(i) paired with flare_severity_on_day(i+lag)
#   lag > 0: flare comes AFTER feature (predictive or concurrent)
#   lag < 0: flare came BEFORE feature (reactive - feature is consequence)
# ============================================================================

# Pre-registered biology-driven lag per feature
PREREGISTERED_FEATURE_LAGS = {
    # Predictive (feature precedes flare)
    'sleep_severity':     2,
    'energy_severity':    1,
    # Concurrent (feature manifests with/right after flare)
    'joint_pain':         1,
    'brain_fog_severity': 1,
    'gi_distress':        1,
    'wellness_severity':  1,
    'fatigue':            2,
    'skin_severity':      2,
    # Reactive (feature appears AFTER flare)
    'phq8_score':        -3,
    'gad7_score':        -3,
}


@dataclass
class DetectedSymptomPattern:
    feature: str
    lag_days: int
    effect_size: float
    p_value: float
    q_value: float
    n_observations: int
    confidence: str


@dataclass
class HypothesisEvidence:
    """Single-hypothesis-level evidence exposed to Conductor for cross-agent corroboration.

    Includes ALL hypotheses tested, regardless of BH survival. Conductor uses these
    sub-threshold signals (e.g., raw_p=0.018 effects suppressed by BH at q=0.10)
    to make cross-agent corroboration decisions.
    """
    feature: str
    lag_days: int
    effect_size: float
    raw_p_value: float
    q_value: float
    n_observations: int
    biology_category: str   # 'predictive' | 'concurrent' | 'reactive'
    survives_fdr: bool      # True if q < fdr_target


_PREDICTIVE_FEATS = {'sleep_severity', 'energy_severity'}
_REACTIVE_FEATS = {'phq8_score', 'gad7_score'}


def _biology_category(feat: str) -> str:
    if feat in _PREDICTIVE_FEATS: return 'predictive'
    if feat in _REACTIVE_FEATS: return 'reactive'
    return 'concurrent'


class SymptomsMoodTriggerDetector:
    def __init__(self, min_days=14,
                 n_permutations=500, binarize_percentile=85,
                 fdr_target=0.10, random_seed=42):
        self.min_days = min_days
        self.n_permutations = n_permutations
        self.binarize_percentile = binarize_percentile
        self.fdr_target = fdr_target
        self.rng = np.random.RandomState(random_seed)
        self.last_n_hypotheses = 0
        self.last_evidence = []

    def detect(self, daily_records, flare_severity_by_date):
        """Returns high-confidence patterns (BH-FDR survivors only). Also populates
        self.last_evidence with all hypothesis-level evidence for raw_hypothesis_evidence().
        """
        if len(daily_records) < self.min_days:
            self.last_n_hypotheses = 0
            self.last_evidence = []
            return []

        records_by_date = {r['date']: r for r in daily_records}
        sorted_dates = sorted(records_by_date.keys())
        candidates = []

        for feat, lag in PREREGISTERED_FEATURE_LAGS.items():
            paired = []
            for i, date in enumerate(sorted_dates):
                target_idx = i + lag
                if target_idx < 0 or target_idx >= len(sorted_dates):
                    continue
                target_date = sorted_dates[target_idx]
                paired.append((date, flare_severity_by_date.get(target_date, 0.0)))

            if len(paired) < self.min_days:
                continue

            flares = np.array([p[1] for p in paired], dtype=float)
            raw = [records_by_date[p[0]].get(feat) for p in paired]
            values = np.array([(np.nan if v is None else v) for v in raw], dtype=float)
            mask = ~np.isnan(values) & ~np.isnan(flares)
            if mask.sum() < self.min_days:
                continue
            v, f = values[mask], flares[mask]
            if np.std(v) < 1e-9 or np.std(f) < 1e-9:
                continue

            threshold = float(np.percentile(v, self.binarize_percentile))
            high = v > threshold
            if high.sum() < 2 or high.sum() > len(v) - 2:
                continue

            diff_obs = float(f[high].mean() - f[~high].mean())
            if diff_obs > 0:
                p_val = self._perm_p(high, f, diff_obs)
            else:
                p_val = 1.0

            candidates.append({
                'feature': feat,
                'label': f'{feat} (>p{self.binarize_percentile})',
                'lag': lag, 'effect': diff_obs, 'p': p_val,
                'n_obs': int(mask.sum()),
            })

        self.last_n_hypotheses = len(candidates)
        if not candidates:
            self.last_evidence = []
            return []

        # BH FDR
        candidates.sort(key=lambda c: c['p'])
        n = len(candidates)
        q_values = [0.0] * n
        running_min = 1.0
        for k in range(n - 1, -1, -1):
            rank = k + 1
            adjusted = candidates[k]['p'] * n / rank
            running_min = min(running_min, adjusted)
            q_values[k] = min(1.0, running_min)

        # Build evidence list (ALL hypotheses, not just survivors)
        self.last_evidence = []
        for k, c in enumerate(candidates):
            q = q_values[k]
            self.last_evidence.append(HypothesisEvidence(
                feature=c['feature'],
                lag_days=c['lag'],
                effect_size=c['effect'],
                raw_p_value=c['p'],
                q_value=q,
                n_observations=c['n_obs'],
                biology_category=_biology_category(c['feature']),
                survives_fdr=(q < self.fdr_target),
            ))

        # Return only BH-FDR survivors
        patterns = []
        for k, c in enumerate(candidates):
            q = q_values[k]
            if q >= self.fdr_target:
                continue
            patterns.append(DetectedSymptomPattern(
                feature=c['label'], lag_days=c['lag'],
                effect_size=c['effect'], p_value=c['p'], q_value=q,
                n_observations=c['n_obs'],
                confidence=self._confidence(q),
            ))

        patterns.sort(key=lambda p: abs(p.effect_size), reverse=True)
        return patterns

    def _perm_p(self, bool_vec, flares, observed_diff):
        perms = np.empty(self.n_permutations)
        for i in range(self.n_permutations):
            shuffled = self.rng.permutation(flares)
            perms[i] = shuffled[bool_vec].mean() - shuffled[~bool_vec].mean()
        return float((perms >= observed_diff).mean())

    def _confidence(self, q):
        if q < 0.01: return 'high'
        if q < 0.05: return 'medium'
        return 'low'

print('SymptomsMoodTriggerDetector defined.')
print(f'  Pre-registered hypotheses: {len(PREREGISTERED_FEATURE_LAGS)}')
print(f'  FDR target: 0.10 (agent-level)')
print(f'  Raw evidence exposed via raw_hypothesis_evidence() for Conductor')

In [ ]:
# ============================================================================
# Section 18: Conductor-Facing Wellness Signature
# ============================================================================
WELLNESS_EVIDENCE_WEIGHTS = {
    'fatigue':             0.20,
    'joint_pain':          0.20,
    'brain_fog_severity':  0.15,
    'wellness_severity':   0.10,
    'energy_severity':     0.10,
    'sleep_severity':      0.10,
    'gi_distress':         0.05,
    'skin_severity':       0.05,
    'phq8_score':          0.025,
    'gad7_score':          0.025,
}


def compute_wellness_signature(
    summary: DailySymptomMoodSummary,
    anomaly_scores: dict,
    detected_patterns: Optional[list] = None,
) -> dict:
    detected_patterns = detected_patterns or []

    triggered_features = set()
    for p in detected_patterns:
        base = p.feature.split(' ')[0]
        if p.confidence in ('high', 'medium'):
            triggered_features.add(base)

    contributions = []
    total_score = 0.0

    for feature, weight in WELLNESS_EVIDENCE_WEIGHTS.items():
        anomaly = anomaly_scores.get(feature, float('nan'))
        normalized = 0.0 if pd.isna(anomaly) else max(0.0, min(3.0, anomaly)) / 3.0
        effective_weight = weight * (1.5 if feature in triggered_features else 1.0)
        total_score += normalized * effective_weight

        contributions.append({
            'feature': feature,
            'anomaly_score': float(anomaly) if not pd.isna(anomaly) else None,
            'normalized': normalized,
            'evidence_weight': weight,
            'effective_weight': effective_weight,
            'is_personal_trigger': feature in triggered_features,
        })

    total_score = min(1.0, total_score)
    contributions.sort(key=lambda c: c['normalized'] * c['effective_weight'], reverse=True)

    return {
        'score': total_score,
        'contributing_factors': contributions,
        'clinical_alerts': [
            f for f, alert in summary.threshold_alerts.items()
            if alert in ('severe', 'moderate', 'moderately_severe')
        ],
        'data_quality_confidence': summary.overall_confidence,
        'flare_score': summary.flare_score,
        'flare_button_pressed': summary.flare_button_pressed,
    }

print('compute_wellness_signature defined.')

In [ ]:
# ============================================================================
# Section 19: SymptomsMoodAgent (Conductor-facing orchestrator)
# ============================================================================
@dataclass
class SymptomsMoodAgentReport:
    n_days_observed: int
    n_flare_events: int
    n_hypotheses_tested: int
    baselines: dict
    today_anomaly_scores: Optional[dict]
    detected_patterns: list
    tracker_activation: dict


class SymptomsMoodAgent:
    def __init__(self,
                 window=14, anomaly_threshold=2.0, personalization_days=25,
                 trigger_min_days=14, n_permutations=500, fdr_target=0.10,
                 memory_store: Optional[MemoryStore] = None,
                 patient_id: str = 'patient_001'):
        self.baseline_tracker = SymptomsMoodRobustTracker(
            window=window,
            anomaly_threshold=anomaly_threshold,
            personalization_days=personalization_days,
        )
        self.trigger_detector = SymptomsMoodTriggerDetector(
            min_days=trigger_min_days,
            n_permutations=n_permutations,
            fdr_target=fdr_target,
        )
        self.memory = memory_store or StubMemoryStore()
        self.patient_id = patient_id
        self.daily_records = []
        self.flare_events = {}
        self._latest_anomalies = None
        self._latest_summary = None

    def observe(self, daily_summary):
        self.baseline_tracker.update(daily_summary)
        self.daily_records.append({
            'date': daily_summary.date,
            **{f: getattr(daily_summary, f, None) for f in ALL_FEATURES},
        })

        self.flare_events[daily_summary.date] = daily_summary.flare_score
        self.memory.add_observation(self.patient_id, daily_summary)

        self._latest_anomalies = self.baseline_tracker.anomaly_scores(daily_summary)
        self._latest_summary = daily_summary
        return {'anomaly_scores': self._latest_anomalies,
                'flare_score': daily_summary.flare_score}

    def observe_flare(self, date, severity):
        """Rarely called - Agent 5 normally produces flares from flare_score."""
        self.flare_events[date] = float(severity)

    def analyze(self):
        flare_dict = dict(self.flare_events)
        for rec in self.daily_records:
            flare_dict.setdefault(rec['date'], 0.0)

        patterns = self.trigger_detector.detect(self.daily_records, flare_dict)
        baselines = self.baseline_tracker.report()
        activation = {f: baselines[f]['n_clean'] >= 3 for f in ALL_FEATURES}
        activation['trigger_detector'] = len(self.daily_records) >= self.trigger_detector.min_days

        return SymptomsMoodAgentReport(
            n_days_observed=len(self.daily_records),
            n_flare_events=sum(1 for s in self.flare_events.values() if s >= 0.5),
            n_hypotheses_tested=self.trigger_detector.last_n_hypotheses,
            baselines=baselines,
            today_anomaly_scores=self._latest_anomalies,
            detected_patterns=patterns,
            tracker_activation=activation,
        )

    def flare_signature(self, daily_summary=None):
        summary = daily_summary or self._latest_summary
        if summary is None:
            return {'score': 0.0, 'contributing_factors': [],
                    'clinical_alerts': [], 'data_quality_confidence': 0.0,
                    'flare_score': 0.0, 'flare_button_pressed': False}
        anomalies = self.baseline_tracker.anomaly_scores(summary)
        report = self.analyze()
        return compute_wellness_signature(summary, anomalies, report.detected_patterns)

    def daily_flare_score(self, daily_summary=None) -> float:
        summary = daily_summary or self._latest_summary
        return summary.flare_score if summary else 0.0

    def jepa_embedding(self, daily_summary=None) -> np.ndarray:
        summary = daily_summary or self._latest_summary
        if summary is None:
            return np.zeros(36, dtype=np.float32)
        return compute_jepa_embedding(summary)

    def raw_hypothesis_evidence(self) -> list:
        """Return all 10 hypothesis-level evidence items from the most recent analyze().

        Use Case: Conductor (Agent 9) consumes this to cross-corroborate sub-threshold
        signals against other agents. A signal at raw_p=0.018 doesn't survive Agent 5's
        own BH FDR at q=0.10, but the Conductor may surface it if e.g. Agent 4 (wearable)
        also reports anomalous sleep on the same days.

        Returns empty list if analyze() has not been called yet.
        """
        if not hasattr(self.trigger_detector, 'last_evidence'):
            return []
        return list(self.trigger_detector.last_evidence)


print('SymptomsMoodAgent defined.')

In [ ]:
# ============================================================================
# Section 20: End-to-end smoke test - live Claude API on hand-crafted transcript
# ============================================================================
SAMPLE_TRANSCRIPT = '''Today's been really tough. Woke up exhausted even after
eight hours of sleep - must have been restless. Knees feel about a 7 today,
hands a bit worse. Can't concentrate on anything, brain's just foggy. Skin's
been itchy near my elbows where the rash was last week. Skipped lunch, no
appetite. Pressed through work but barely. Mood is low, anxious about
tomorrow's meeting.'''

print('SAMPLE TRANSCRIPT:')
print(SAMPLE_TRANSCRIPT)
print()

if not _ANTHROPIC_AVAILABLE:
    print('SKIPPED: anthropic SDK not installed. Run: pip install anthropic')
elif not os.environ.get('ANTHROPIC_API_KEY'):
    print('SKIPPED: ANTHROPIC_API_KEY not set in .env')
else:
    composite = CompositeSymptomSource(voice=VoiceTranscriptSource())
    today = datetime.now().strftime('%Y-%m-%d')

    summary = process_symptom_day(
        patient_id='smoke_test_001',
        target_date=today,
        disease='SLE',
        transcript=SAMPLE_TRANSCRIPT,
        composite_source=composite,
    )

    print(f'Date:      {summary.date}')
    print(f'Patient:   {summary.patient_id}')
    print(f'Disease:   {summary.disease}')
    print()
    print('Extracted symptoms (from voice transcript via Claude):')
    for feat in SYMPTOM_FEATURES:
        v = getattr(summary, feat)
        pct = summary.percentiles.get(feat)
        if v is not None and pct is not None:
            print(f'  {feat:25s}: {v}/10  (pct={pct:.0%})')
    print()
    print('Mood:')
    print(f'  PHQ-8: {summary.phq8_score}    (alert={summary.threshold_alerts.get("phq8")})')
    print(f'  GAD-7: {summary.gad7_score}    (alert={summary.threshold_alerts.get("gad7")})')
    print()
    print(f'NLP enrichment:')
    print(f'  emotional_valence: {summary.emotional_valence}')
    print(f'  new_symptom_mentions: {summary.new_symptom_mentions}')
    print()
    print(f'DAILY FLARE SCORE (-> Conductor): {summary.flare_score:.3f}')
    print(f'Overall confidence: {summary.overall_confidence:.0%}')
    print()
    print('JEPA embedding (first 12 dims of 36):')
    emb = compute_jepa_embedding(summary)
    print(f'  {emb[:12]}')
    if summary.errors:
        print(f'\nErrors: {summary.errors}')

In [ ]:
# ============================================================================
# Section 21: Layer 3 Validation - 4 tests with corrected biology
#
# Test 1: Concurrent fatigue trigger (lag=+2)
# Test 2: Predictive sleep trigger (lag=+2)
# Test 3: Reactive PHQ-8 trigger (lag=-3, true bidirectional case)
# Test 4: Negative control (no trigger)
#
# Expected behavior at FDR=0.10:
#   Test 1: detects with raw_p < 0.010 (concurrent + sufficient signal density)
#   Test 2: may not detect (sleep trigger threshold = 7.0, baseline (4.0, 1.8)
#           produces only ~3 trigger activations / 60 days, signal diluted)
#   Test 3: real signal exists (raw_p ~ 0.018) but BH-suppressed at q=0.18.
#           Conductor recovers this via raw_hypothesis_evidence().
#   Test 4: clean (no patterns)
# ============================================================================
class SyntheticSymptomPatient:
    """Generates daily symptom records with one configured trigger.

    Modes:
      Predictive/concurrent (trigger_lag_days >= 0):
        High feature on day i -> flare scheduled on day i + lag
      Reactive (trigger_lag_days < 0):
        Random baseline-rate flares; each flare boosts trigger feature
        on day flare_day + abs(lag)
    """

    BASELINE_DISTS = {
        'fatigue':             (4.5, 1.8),
        'joint_pain':          (4.0, 1.8),
        'brain_fog_severity':  (3.5, 1.7),
        'gi_distress':         (2.5, 1.5),
        'skin_severity':       (2.0, 1.3),
        'sleep_severity':      (4.0, 1.8),
        'energy_severity':     (4.0, 1.7),
        'wellness_severity':   (4.0, 1.6),
    }
    MOOD_DISTS = {
        'phq8_score': (7.5, 4.5),
        'gad7_score': (6.0, 4.0),
    }
    TRIGGER_THRESHOLDS = {
        'fatigue': 7.0, 'joint_pain': 7.0, 'sleep_severity': 7.0,
        'brain_fog_severity': 6.5, 'gi_distress': 6.0, 'skin_severity': 5.5,
        'energy_severity': 6.5, 'wellness_severity': 7.0,
    }
    REACTIVE_BOOST = 5.0

    def __init__(self,
                 trigger_feature=None,
                 trigger_lag_days=2,
                 trigger_flare_probability=0.85,
                 baseline_flare_probability=0.10,
                 random_seed=42):
        self.trigger_feature = trigger_feature
        self.trigger_lag_days = trigger_lag_days
        self.trigger_flare_probability = trigger_flare_probability
        self.baseline_flare_probability = baseline_flare_probability
        self.rng = random_mod.Random(random_seed)
        self.np_rng = np.random.RandomState(random_seed)

    def _is_trigger_active(self, features):
        if self.trigger_feature is None or self.trigger_lag_days < 0:
            return False
        t = self.TRIGGER_THRESHOLDS.get(self.trigger_feature)
        v = features.get(self.trigger_feature)
        return t is not None and v is not None and v > t

    def generate(self, n_days=60, start_date='2026-03-01'):
        records, scheduled_flares = [], {}
        start = pd.Timestamp(start_date)

        for i in range(n_days):
            date = (start + pd.Timedelta(days=i)).strftime('%Y-%m-%d')
            features = {'date': date}
            for feat, (mean, std) in self.BASELINE_DISTS.items():
                v = self.np_rng.normal(mean, std)
                features[feat] = max(0.0, min(10.0, v))
            if pd.Timestamp(date).day_of_week == 0:
                for feat, (mean, std) in self.MOOD_DISTS.items():
                    cap = 24 if feat == 'phq8_score' else 21
                    features[feat] = max(0.0, min(cap, self.np_rng.normal(mean, std)))
            else:
                if records and 'phq8_score' in records[-1]:
                    features['phq8_score'] = records[-1]['phq8_score']
                    features['gad7_score'] = records[-1]['gad7_score']

            if self.trigger_lag_days >= 0:
                flare_prob = (self.trigger_flare_probability
                              if self._is_trigger_active(features)
                              else self.baseline_flare_probability)
            else:
                flare_prob = self.baseline_flare_probability

            if self.rng.random() < flare_prob:
                if self.trigger_lag_days >= 0:
                    flare_idx = i + self.trigger_lag_days
                else:
                    flare_idx = i
                if 0 <= flare_idx < n_days:
                    flare_date = (start + pd.Timedelta(days=flare_idx)).strftime('%Y-%m-%d')
                    severity = float(self.np_rng.uniform(0.5, 0.9))
                    scheduled_flares[flare_date] = max(scheduled_flares.get(flare_date, 0.0), severity)

            records.append(features)

        # Reactive mode: boost trigger feature on day (flare_day + |lag|)
        if self.trigger_lag_days < 0 and self.trigger_feature is not None:
            offset = abs(self.trigger_lag_days)
            for flare_date_str, severity in scheduled_flares.items():
                flare_day = (pd.Timestamp(flare_date_str) - start).days
                target_day = flare_day + offset
                if 0 <= target_day < n_days and target_day < len(records):
                    boost = self.REACTIVE_BOOST * severity
                    current = records[target_day].get(self.trigger_feature, 0.0)
                    if self.trigger_feature in self.MOOD_DISTS:
                        cap = 24 if self.trigger_feature == 'phq8_score' else 21
                    else:
                        cap = 10.0
                    records[target_day][self.trigger_feature] = min(cap, current + boost)

        return records, scheduled_flares


class _SymptomSummaryLike:
    def __init__(self, d):
        self.date = d['date']
        self.patient_id = 'synthetic'
        self.disease = 'Mixed'
        for feat in ALL_FEATURES:
            setattr(self, feat, d.get(feat))
        self.flare_score = 0.0
        self.overall_confidence = 1.0
        self.threshold_alerts = {}
        self.percentiles = {}
        self.emotional_valence = None
        self.new_symptom_mentions = []


def _run_test(label, patient, expectation):
    records, flares = patient.generate(n_days=60)
    agent = SymptomsMoodAgent(n_permutations=500, patient_id='synthetic')

    for r in records:
        summary = _SymptomSummaryLike(r)
        summary.flare_score = flares.get(r['date'], 0.0)
        agent.observe(summary)

    report = agent.analyze()
    print(f'\n{label}')
    print(f'  Expectation:  {expectation}')
    print(f'  Observed:     {report.n_days_observed} days, {report.n_flare_events} flare events')
    print(f'  Hypotheses:   {report.n_hypotheses_tested}')
    print(f'  Patterns ({len(report.detected_patterns)} found):')
    for p in report.detected_patterns:
        print(f'    {p.feature:30s} lag={p.lag_days:+d}d  effect={p.effect_size:+.3f}'
              f'  p={p.p_value:.3f}  q={p.q_value:.3f}  [{p.confidence}]')

    # Show raw evidence for Conductor consumption
    evidence = agent.raw_hypothesis_evidence()
    sub_threshold = [e for e in evidence if not e.survives_fdr and e.raw_p_value < 0.05]
    if sub_threshold:
        print(f'  Sub-threshold evidence (raw_p < 0.05, for Conductor cross-corroboration):')
        for e in sub_threshold:
            print(f'    {e.feature:25s} lag={e.lag_days:+d}d  effect={e.effect_size:+.3f}'
                  f'  raw_p={e.raw_p_value:.3f}  q={e.q_value:.3f}  [{e.biology_category}]')


print('=' * 70)
print('Agent 5 Layer 3 validation - 4 tests')
print('=' * 70)

_run_test(
    'TEST 1: Concurrent (fatigue trigger, lag=+2)',
    SyntheticSymptomPatient(trigger_feature='fatigue', trigger_lag_days=2, random_seed=42),
    'fatigue (>p85) at lag=+2d  [concurrent]',
)
_run_test(
    'TEST 2: Predictive (sleep trigger, lag=+2)',
    SyntheticSymptomPatient(trigger_feature='sleep_severity', trigger_lag_days=2, random_seed=43),
    'sleep_severity (>p85) at lag=+2d  [predictive]',
)
_run_test(
    'TEST 3: Reactive (phq8 trigger, lag=-3, true bidirectional)',
    SyntheticSymptomPatient(trigger_feature='phq8_score', trigger_lag_days=-3, random_seed=44),
    'phq8_score (>p85) at lag=-3d  [reactive - may be BH-suppressed, exposed via raw_hypothesis_evidence]',
)
_run_test(
    'TEST 4: Negative control',
    SyntheticSymptomPatient(trigger_feature=None, random_seed=45),
    'Zero patterns at [high]; ideally zero patterns at all',
)

print('\n' + '=' * 70)
print('Validation complete. See notebook header for limitations.')
print('=' * 70)